# Notebook 88 — A5 Contextual Safe-Cleanup Model Training

**Runtime:** GPU. Select a T4 GPU, then **Runtime → Run all**.

## Purpose

Train a development-only contextual model which starts from the frozen **A1** page and may clean an A1-preserved ambiguous pixel only when observable local evidence supports **ANNOTATION** strongly enough.

The operational rule is deliberately asymmetric:

- **A5 CLEAN** → replace the A1 pixel with the frozen local paper estimate.
- **A5 KEEP / UNCERTAIN** → retain the exact A1 byte.
- Outside the authoritative reviewed removal mask → no change.

## Scientific boundary

Notebook 88 uses the completed Notebook 87 paired-HITL development run as labelled development evidence. Clean truth is used only through Notebook 87's post-lock labels. Clean image pixels and truth-derived arrays are prohibited inference channels.

The notebook:

1. resolves one completed Notebook 87 run;
2. verifies its review and prediction locks;
3. builds observable 33×33 tensors for A1-preserved ambiguous pixels;
4. trains compact CNNs under source-group out-of-fold separation;
5. fits and persists fold-specific isotonic calibration;
6. scores every eligible held-out pixel;
7. applies conservative CLEAN / KEEP / UNCERTAIN rules;
8. creates complete A1-versus-A5 pages;
9. reports annotation cleanup, genuine-print loss, catastrophic concentrations and page/source stability;
10. packages all training artefacts needed for a later freeze step.

**This is development training, not independent validation or production authorisation.**

In [1]:
# ============================================================
# 0. Mount Drive and install dependencies
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision matplotlib joblib

Mounted at /content/drive


In [3]:
# ============================================================
# 1. Configuration, constants and run identity
# ============================================================

import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform, gc, shutil, time

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
N87_ROOT = ROOT / 'paired_hitl_development' / 'runs'
OUTPUT_ROOT = ROOT / 'a5_contextual_safe_cleanup_v88'

# Optional exact completed Notebook 87 run. Leave None to use the latest complete run.
SOURCE_N87_RUN_OVERRIDE = None

SEED = 42
PATCH = 33
HALF = PATCH // 2
BATCH = 512
EPOCHS = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_PER_PAGE_CLASS = 25_000
BOOTSTRAP_RESAMPLES = 10_000

# The model estimates p(annotation-safe-to-clean).
# These are predeclared development operating points, not production thresholds.
CLEAN_THRESHOLDS = [0.995, 0.990, 0.980, 0.950]
PRIMARY_CLEAN_THRESHOLD = 0.995
MIN_CLEAN_RUN = 2

# Labels inherited from Notebook 87 truth construction.
LABEL_PRINT = 0
LABEL_ANNOTATION = 1
LABEL_UNCERTAIN = 2

# Observable-only channels. The exact ordered list is saved and hashed.
CHANNELS = [
    'observed',
    'local_normalised',
    'reviewed_mask',
    'mask_depth',
    'ambiguous',
    'p_annotation',
    'p_print',
    'edge',
    'horizontal_support',
    'vertical_support',
    'compact_support',
]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Notebook 88 requires a GPU runtime. Select a T4 GPU and rerun.')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_notebook88_a5_training')
RUN = OUTPUT_ROOT / RUN_ID
INPUTS = RUN / 'inputs'
TABLES = RUN / 'tables'
MODELS = RUN / 'models'
CALIBRATORS = RUN / 'calibrators'
PAGES = RUN / 'pages'
AUDIT = RUN / 'audit'
VISUALS = RUN / 'visuals'
UPLOAD = RUN / 'upload'

for folder in [RUN, INPUTS, TABLES, MODELS, CALIBRATORS, PAGES, AUDIT, VISUALS, UPLOAD]:
    folder.mkdir(parents=True, exist_ok=False)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def object_hash(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str, separators=(',', ':')).encode()
    ).hexdigest()


def array_hash(array):
    a = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()

print('Device:', DEVICE, torch.cuda.get_device_name(0))
print('Output:', RUN)

Device: cuda Tesla T4
Output: /content/drive/MyDrive/MusicAnnotationProject/a5_contextual_safe_cleanup_v88/20261121_231807_notebook88_a5_training


In [4]:
# ============================================================
# 2. Resolve and verify one completed Notebook 87 source run
# ============================================================

N87_REQUIRED = [
    'prediction_lock.json',
    'notebook87_verdict.json',
    'tables/paired_hitl_ambiguous_truth.parquet',
    'tables/truth_scores_by_page.csv',
    'tables/truth_scores_pooled.csv',
]


def complete_n87_runs(root):
    hits = []
    if root.exists():
        for path in root.iterdir():
            if path.is_dir() and all((path / rel).exists() for rel in N87_REQUIRED):
                hits.append(path)
    return sorted(hits, key=lambda p: p.stat().st_mtime)


if SOURCE_N87_RUN_OVERRIDE is not None:
    SOURCE_N87 = Path(SOURCE_N87_RUN_OVERRIDE)
    missing = [rel for rel in N87_REQUIRED if not (SOURCE_N87 / rel).exists()]
    require(not missing, f'Notebook 87 source override is incomplete: {missing}')
else:
    runs = complete_n87_runs(N87_ROOT)
    require(runs, f'No complete Notebook 87 run found below {N87_ROOT}')
    SOURCE_N87 = runs[-1]

prediction_lock = json.loads((SOURCE_N87 / 'prediction_lock.json').read_text())
verdict87 = json.loads((SOURCE_N87 / 'notebook87_verdict.json').read_text())

require(
    prediction_lock.get('state') == 'CLOSED_BEFORE_CLEAN_TRUTH',
    'Notebook 87 prediction lock is not closed before clean truth.',
)
require(
    prediction_lock.get('clean_truth_read') is False,
    'Notebook 87 prediction lock reports truth leakage.',
)
require(
    verdict87.get('automatic_promotion') is False,
    'Unexpected Notebook 87 promotion state.',
)

review_manifest_path = ROOT / 'paired_hitl_development' / 'locks' / 'paired_hitl_review_manifest.csv'
review_lock_path = ROOT / 'paired_hitl_development' / 'locks' / 'paired_hitl_review_lock.json'
require(review_manifest_path.exists(), f'Missing review manifest: {review_manifest_path}')
require(review_lock_path.exists(), f'Missing review lock: {review_lock_path}')

review_manifest = pd.read_csv(review_manifest_path, dtype=str).fillna('')
truth_pixels = pd.read_parquet(SOURCE_N87 / 'tables' / 'paired_hitl_ambiguous_truth.parquet')
truth_scores87 = pd.read_csv(SOURCE_N87 / 'tables' / 'truth_scores_by_page.csv')

required_truth = {
    'page', 'y', 'x', 'p_annotation', 'p_print',
    'truth_print', 'truth_annotation', 'truth_uncertain',
}
require(required_truth.issubset(truth_pixels.columns),
        f'Notebook 87 truth table missing: {sorted(required_truth-set(truth_pixels.columns))}')
require(not truth_pixels.duplicated(['page','y','x']).any(),
        'Notebook 87 truth table has duplicate page/y/x rows.')

# Snapshot the exact source records used by this run.
for source, target in [
    (SOURCE_N87 / 'prediction_lock.json', INPUTS / 'notebook87_prediction_lock.json'),
    (SOURCE_N87 / 'notebook87_verdict.json', INPUTS / 'notebook87_verdict.json'),
    (review_manifest_path, INPUTS / 'paired_hitl_review_manifest.csv'),
    (review_lock_path, INPUTS / 'paired_hitl_review_lock.json'),
]:
    shutil.copy2(source, target)

print('Notebook 87 source:', SOURCE_N87)
print('Truth rows:', len(truth_pixels))
print('Truth pages:', truth_pixels.page.astype(str).nunique())

RuntimeError: No complete Notebook 87 run found below /content/drive/MyDrive/MusicAnnotationProject/paired_hitl_development/runs

In [4]:
# ============================================================
# 3. Resolve page roles and build a fail-closed source-group map
# ============================================================

review_manifest['page'] = review_manifest.page.astype(str)
truth_pixels['page'] = truth_pixels.page.astype(str)

if 'source_group' not in review_manifest.columns:
    review_manifest['source_group'] = ''

# Missing source groups are not allowed to leak pages across folds.
# Each missing group becomes a page-specific group, which is conservative.
review_manifest['source_group'] = review_manifest.apply(
    lambda r: str(r.source_group).strip() if str(r.source_group).strip()
    else f'PAGE_ONLY__{r.page}',
    axis=1,
)

page_meta = review_manifest.set_index('page')
truth_pages = sorted(truth_pixels.page.unique())
missing_pages = sorted(set(truth_pages) - set(page_meta.index))
require(not missing_pages, f'Truth pages absent from review manifest: {missing_pages}')

# Pages with no eligible truth rows do not enter model fitting.
role_rows = []
for page in review_manifest.page:
    n = int((truth_pixels.page == page).sum())
    role_rows.append({
        'page': page,
        'source_group': page_meta.loc[page, 'source_group'],
        'eligible_truth_rows': n,
        'model_role': 'DEVELOPMENT_OOF' if n > 0 else 'ZERO_EXPOSURE_CONTROL',
    })
roles = pd.DataFrame(role_rows)
roles.to_csv(INPUTS / 'page_role_manifest.csv', index=False)

print(roles.model_role.value_counts())
print('Development source groups:', roles.loc[roles.model_role=='DEVELOPMENT_OOF','source_group'].nunique())

model_role
DEVELOPMENT_OOF          56
ZERO_EXPOSURE_CONTROL    27
Name: count, dtype: int64
Development source groups: 21


In [5]:
# ============================================================
# 4. Observable channel construction
#
# Clean truth is not loaded here. Labels come only from the already
# post-lock Notebook 87 truth table.
# ============================================================


def read_gray(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(image is not None, f'Unable to read image: {path}')
    return image


def reconcile_zero_authority_canvas(array, target_shape, page, name):
    if array.shape == target_shape:
        return array
    h, w = array.shape
    th, tw = target_shape
    dh, dw = th-h, tw-w
    require(dh in (0,1) and dw in (0,1),
            f'{page}: {name} shape {array.shape} cannot safely reconcile to {target_shape}')
    fixed = np.zeros(target_shape, dtype=array.dtype)
    fixed[:h,:w] = array
    return fixed


def background_surface(observed, mask, sigma=45.0):
    visible = (~mask) & (observed >= 200)
    numerator = cv2.GaussianBlur(observed.astype(np.float32)*visible, (0,0), sigma)
    denominator = cv2.GaussianBlur(visible.astype(np.float32), (0,0), sigma)
    fallback = cv2.GaussianBlur(observed, (0,0), sigma)
    return np.clip(
        np.where(denominator > 0.02, numerator/np.maximum(denominator,1e-4), fallback),
        0, 255,
    ).astype(np.uint8)


def support_maps(observed, mask, band=4):
    dark = (observed < 190) & (~mask)
    horizontal = cv2.morphologyEx(
        dark.astype(np.uint8)*255, cv2.MORPH_OPEN, np.ones((1,35),np.uint8)
    ) > 0
    vertical = cv2.morphologyEx(
        dark.astype(np.uint8)*255, cv2.MORPH_OPEN, np.ones((15,1),np.uint8)
    ) > 0
    n, labels, stats, _ = cv2.connectedComponentsWithStats(dark.astype(np.uint8), 8)
    compact = np.zeros_like(dark)
    for component_id in range(1,n):
        x,y,w,h,area = stats[component_id]
        if 3 <= area <= 650 and w <= 40 and h <= 40:
            compact[labels == component_id] = True
    kernel = np.ones((2*band+1,2*band+1),np.uint8)
    return (
        cv2.dilate(horizontal.astype(np.uint8),kernel)>0,
        cv2.dilate(vertical.astype(np.uint8),kernel)>0,
        cv2.dilate(compact.astype(np.uint8),kernel)>0,
    )


runtime = {}
input_rows = []
canvas_repairs = []

for page in truth_pages:
    row = page_meta.loc[page]
    page_dir = SOURCE_N87 / 'pages' / page
    require(page_dir.exists(), f'{page}: Notebook 87 page directory missing')

    observed = read_gray(page_dir / 'observed.png')
    seed = read_gray(page_dir / 'reviewed_seed.png')
    ambiguous = read_gray(page_dir / 'ambiguous.png') > 0
    a1 = read_gray(page_dir / 'A1_BASELINE.png')

    seed_original_shape = seed.shape
    seed = reconcile_zero_authority_canvas(seed, observed.shape, page, 'reviewed seed')
    if seed.shape != seed_original_shape:
        canvas_repairs.append({
            'page': page,
            'original_shape': str(seed_original_shape),
            'target_shape': str(observed.shape),
            'authority_pixels_added': 0,
        })

    with np.load(page_dir / 'arrays.npz') as z:
        p_annotation = z['p_annotation'].astype(np.float32)
        p_print = z['p_print'].astype(np.float32)
        ambiguous_npz = z['ambiguous'] > 0

    require(observed.shape == seed.shape == ambiguous.shape == a1.shape,
            f'{page}: observable page arrays have inconsistent shapes')
    require(p_annotation.shape == observed.shape and p_print.shape == observed.shape,
            f'{page}: probability arrays have inconsistent shapes')
    require(np.array_equal(ambiguous, ambiguous_npz),
            f'{page}: ambiguous image/NPZ mismatch')

    reviewed_mask = seed > 0
    paper = background_surface(observed, reviewed_mask)
    local_normalised = cv2.divide(observed, np.maximum(paper,1), scale=255)
    mask_depth = cv2.distanceTransform(reviewed_mask.astype(np.uint8), cv2.DIST_L2, 5)
    edge = cv2.Canny(observed,70,170) > 0
    horizontal, vertical, compact = support_maps(observed, reviewed_mask)

    # Eligible coordinates are exactly the Notebook 87 ambiguous truth rows.
    q = truth_pixels[truth_pixels.page == page].copy()
    ys = q.y.to_numpy(int)
    xs = q.x.to_numpy(int)
    require((ys>=0).all() and (xs>=0).all() and
            (ys<observed.shape[0]).all() and (xs<observed.shape[1]).all(),
            f'{page}: truth coordinate outside observable arrays')
    require(ambiguous[ys,xs].all(), f'{page}: truth table contains non-ambiguous coordinates')

    runtime[page] = {
        'observed': observed,
        'a1': a1,
        'paper': paper,
        'reviewed_mask': reviewed_mask,
        'ambiguous': ambiguous,
        'local_normalised': local_normalised,
        'mask_depth': mask_depth,
        'p_annotation': p_annotation,
        'p_print': p_print,
        'edge': edge,
        'horizontal_support': horizontal,
        'vertical_support': vertical,
        'compact_support': compact,
    }

    input_rows.append({
        'page': page,
        'source_group': row.source_group,
        'height': observed.shape[0],
        'width': observed.shape[1],
        'eligible_rows': len(q),
        'truth_print': int(q.truth_print.astype(bool).sum()),
        'truth_annotation': int(q.truth_annotation.astype(bool).sum()),
        'truth_uncertain': int(q.truth_uncertain.astype(bool).sum()),
        'observed_sha256': sha256_file(page_dir/'observed.png'),
        'arrays_sha256': sha256_file(page_dir/'arrays.npz'),
        'a1_sha256': sha256_file(page_dir/'A1_BASELINE.png'),
    })

pd.DataFrame(input_rows).to_csv(AUDIT/'input_and_label_audit.csv', index=False)
pd.DataFrame(canvas_repairs).to_csv(AUDIT/'reviewed_seed_canvas_repairs.csv', index=False)

channel_schema = {
    'patch': PATCH,
    'ordered_channels': CHANNELS,
    'dtype': 'float32',
    'normalisation': {
        'observed': 'divide_255',
        'local_normalised': 'divide_255',
        'p_annotation': 'native_0_1',
        'p_print': 'native_0_1',
        'mask_depth': 'divide_by_patch_radius_clipped_0_1',
        'binary_channels': '0_or_1',
    },
    'padding': 'reflect',
    'truth_derived_channels': [],
}
(INPUTS/'channel_schema.json').write_text(json.dumps(channel_schema,indent=2))
print(pd.DataFrame(input_rows)[['eligible_rows','truth_print','truth_annotation','truth_uncertain']].sum())

eligible_rows       324592
truth_print         191468
truth_annotation    119421
truth_uncertain      13703
dtype: int64


In [12]:
# ============================================================
# 5. Build labels, the complete eligible roster and stable
#    page-balanced training sample
# ============================================================

rows = []
for r in truth_pixels.itertuples(index=False):
    flags = [bool(r.truth_print), bool(r.truth_annotation), bool(r.truth_uncertain)]
    require(sum(flags) == 1, f'{r.page}/{r.y}/{r.x}: truth flags are not one-hot')
    label = LABEL_PRINT if flags[0] else LABEL_ANNOTATION if flags[1] else LABEL_UNCERTAIN
    rows.append({
        'page': str(r.page),
        'source_group': str(page_meta.loc[str(r.page),'source_group']),
        'y': int(r.y),
        'x': int(r.x),
        'label': label,
        'p_annotation_existing': float(r.p_annotation),
        'p_print_existing': float(r.p_print),
    })

eligible = pd.DataFrame(rows).sort_values(['source_group','page','y','x']).reset_index(drop=True)
eligible['candidate_id'] = np.arange(len(eligible), dtype=np.int64)

# UNCERTAIN remains in the complete roster for evaluation but is excluded
# from the supervised binary CLEAN-vs-KEEP loss.
rng = np.random.default_rng(SEED)
sampled_parts = []
for page, group in eligible[eligible.label.isin([LABEL_PRINT,LABEL_ANNOTATION])].groupby('page'):
    for label, cls in group.groupby('label'):
        if len(cls) > MAX_PER_PAGE_CLASS:
            take = rng.choice(cls.index.to_numpy(), MAX_PER_PAGE_CLASS, replace=False)
            cls = cls.loc[take]
        sampled_parts.append(cls)

stable = pd.concat(sampled_parts, ignore_index=True).sort_values(
    ['source_group','page','y','x']
).reset_index(drop=True)
stable['sample_id'] = np.arange(len(stable), dtype=np.int64)

require(not stable.empty, 'Stable training sample is empty.')
require(stable.label.nunique() == 2, 'Stable training sample does not contain PRINT and ANNOTATION.')

# Source-group OOF where possible. Page-only fallback groups remain isolated.
groups = stable.source_group.astype(str)
unique_groups = groups.nunique()
require(unique_groups >= 3, 'At least three source groups/page groups are required.')
splitter = LeaveOneGroupOut() if unique_groups <= 10 else GroupKFold(n_splits=min(10,unique_groups))

stable['fold'] = -1
group_to_fold = {}
split_rows = []
for fold_id, (train_idx, test_idx) in enumerate(splitter.split(stable, stable.label, groups), start=1):
    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])
    overlap = train_groups & test_groups
    require(not overlap, f'Fold {fold_id}: source-group leakage {sorted(overlap)}')
    stable.loc[test_idx,'fold'] = fold_id
    for group_name in test_groups:
        require(group_name not in group_to_fold, f'Source group assigned twice: {group_name}')
        group_to_fold[group_name] = fold_id
    split_rows.append({
        'fold': fold_id,
        'train_groups': len(train_groups),
        'test_groups': len(test_groups),
        'test_pages': stable.iloc[test_idx].page.nunique(),
        'test_rows': len(test_idx),
        'test_source_groups': '|'.join(sorted(test_groups)),
        'overlap_groups': 0,
    })

# Fill any source groups entirely missing from stable training sample with a default fold (e.g. fold 1)
for g in eligible.source_group.unique():
    if g not in group_to_fold:
        group_to_fold[g] = 1

eligible['fold'] = eligible.source_group.map(group_to_fold)
require((stable.fold > 0).all(), 'Incomplete stable fold assignment.')
require(eligible.fold.notna().all(), 'Incomplete eligible fold assignment.')
eligible['fold'] = eligible.fold.astype(int)

stable.to_parquet(TABLES/'stable_training_sample.parquet', index=False)
eligible.to_parquet(TABLES/'eligible_pixel_manifest.parquet', index=False)
pd.DataFrame(split_rows).to_csv(AUDIT/'source_group_split_audit.csv', index=False)
pd.DataFrame([{'source_group':g,'fold':f} for g,f in sorted(group_to_fold.items())]).to_csv(
    INPUTS/'fold_map.csv', index=False
)

print('Stable supervised rows:', len(stable))
print(stable.label.value_counts().rename(index={0:'PRINT_KEEP',1:'ANNOTATION_CLEAN'}))
print('Complete eligible rows:', len(eligible))
print('Folds:', sorted(stable.fold.unique()))

Stable supervised rows: 310889
label
PRINT_KEEP          191468
ANNOTATION_CLEAN    119421
Name: count, dtype: int64
Complete eligible rows: 324592
Folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


In [2]:
# ============================================================
# 6. Tensor builder and compact contextual CNN
# ============================================================

padded = {}
for page, z in runtime.items():
    depth = np.clip(z['mask_depth']/max(HALF,1), 0, 1).astype(np.float32)
    channel_arrays = {
        'observed': z['observed'].astype(np.float32)/255.0,
        'local_normalised': z['local_normalised'].astype(np.float32)/255.0,
        'reviewed_mask': z['reviewed_mask'].astype(np.float32),
        'mask_depth': depth,
        'ambiguous': z['ambiguous'].astype(np.float32),
        'p_annotation': np.clip(z['p_annotation'],0,1).astype(np.float32),
        'p_print': np.clip(z['p_print'],0,1).astype(np.float32),
        'edge': z['edge'].astype(np.float32),
        'horizontal_support': z['horizontal_support'].astype(np.float32),
        'vertical_support': z['vertical_support'].astype(np.float32),
        'compact_support': z['compact_support'].astype(np.float32),
    }
    padded[page] = {
        key: np.pad(channel_arrays[key], HALF, mode='reflect')
        for key in CHANNELS
    }

page_list = sorted(runtime)
page_index = {page:i for i,page in enumerate(page_list)}


def encode(frame):
    return (
        frame.page.map(page_index).to_numpy(np.int16),
        frame.y.to_numpy(np.int32),
        frame.x.to_numpy(np.int32),
        frame.label.to_numpy(np.int8),
    )

stable_pid, stable_y, stable_x, stable_label = encode(stable)
eligible_pid, eligible_y, eligible_x, eligible_label = encode(eligible)


def get_patch(pid, y, x):
    page = page_list[int(pid)]
    return np.stack([
        padded[page][key][y:y+PATCH, x:x+PATCH]
        for key in CHANNELS
    ]).astype(np.float32)


class ContextPatchDataset(Dataset):
    def __init__(self, indices, pids, ys, xs, labels):
        self.indices = np.asarray(indices)
        self.pids = pids
        self.ys = ys
        self.xs = xs
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = int(self.indices[item])
        patch = get_patch(self.pids[index], self.ys[index], self.xs[index])
        return (
            torch.from_numpy(patch),
            torch.tensor(int(self.labels[index]), dtype=torch.long),
        )


class A5ContextCNN(nn.Module):
    def __init__(self, channels=len(CHANNELS)):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(channels, 24, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(24),
            nn.MaxPool2d(2),
            nn.Conv2d(24, 48, 3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(48),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(64, 2)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))


def worker_init(worker_id):
    random.seed(SEED + worker_id)
    np.random.seed(SEED + worker_id)


def make_loader(indices, pids, ys, xs, labels, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)
    return DataLoader(
        ContextPatchDataset(indices,pids,ys,xs,labels),
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=True,
        generator=generator,
        worker_init_fn=worker_init,
        persistent_workers=False,
    )


def fit_model(train_indices):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = A5ContextCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    counts = np.bincount(stable_label[train_indices], minlength=2)
    weights = torch.tensor(counts.sum()/np.maximum(counts,1), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    history = []
    for epoch in range(EPOCHS):
        model.train(); total = 0.0
        for xb, yb in make_loader(train_indices,stable_pid,stable_y,stable_x,stable_label,True,SEED):
            xb = xb.to(DEVICE,non_blocking=True); yb = yb.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb),yb)
            loss.backward(); optimizer.step()
            total += float(loss.detach())*len(yb)
        mean_loss = total/max(len(train_indices),1)
        history.append(mean_loss)
        print(f'  epoch {epoch+1:02d}/{EPOCHS}: loss={mean_loss:.6f}')
    return model, history


def predict_annotation(model, indices, pids, ys, xs, labels):
    output = []
    model.eval()
    with torch.no_grad():
        for xb, _ in make_loader(indices,pids,ys,xs,labels,False,SEED):
            probability = torch.softmax(model(xb.to(DEVICE,non_blocking=True)),1)[:,LABEL_ANNOTATION]
            output.extend(probability.cpu().numpy())
    return np.asarray(output,dtype=np.float64)

print('Tensor builder ready:', PATCH, 'x', PATCH, 'channels:', len(CHANNELS))

NameError: name 'runtime' is not defined

In [ ]:
# ============================================================
# 7. Source-group OOF model training and all-pixel scoring
# ============================================================

raw_stable = np.full(len(stable), np.nan, dtype=np.float64)
raw_eligible = np.full(len(eligible), np.nan, dtype=np.float64)
model_rows = []
history_rows = []

for fold_id in sorted(stable.fold.unique()):
    print('\n' + '='*80)
    print('FOLD', fold_id)
    train_idx = np.where(stable.fold.to_numpy() != fold_id)[0]
    test_idx = np.where(stable.fold.to_numpy() == fold_id)[0]
    eligible_idx = np.where(eligible.fold.to_numpy() == fold_id)[0]

    require(stable_label[train_idx].min()==0 and stable_label[train_idx].max()==1,
            f'Fold {fold_id}: training side lacks a class')

    started = time.time()
    model, history = fit_model(train_idx)
    raw_stable[test_idx] = predict_annotation(model,test_idx,stable_pid,stable_y,stable_x,stable_label)
    raw_eligible[eligible_idx] = predict_annotation(
        model,eligible_idx,eligible_pid,eligible_y,eligible_x,eligible_label
    )

    checkpoint = MODELS / f'fold{int(fold_id)}_seed{SEED}_a5_context_cnn.pt'
    torch.save(model.state_dict(), checkpoint)
    model_rows.append({
        'fold': int(fold_id),
        'checkpoint': str(checkpoint),
        'checkpoint_sha256': sha256_file(checkpoint),
        'train_rows': len(train_idx),
        'stable_test_rows': len(test_idx),
        'eligible_test_rows': len(eligible_idx),
        'seconds': time.time()-started,
    })
    history_rows.extend({
        'fold': int(fold_id), 'epoch': epoch+1, 'mean_loss': loss
    } for epoch,loss in enumerate(history))

    del model
    gc.collect()
    torch.cuda.empty_cache()

require(np.isfinite(raw_stable).all(), 'Incomplete stable OOF predictions.')
require(np.isfinite(raw_eligible).all(), 'Incomplete all-eligible OOF predictions.')

np.save(TABLES/'stable_raw_oof.npy',raw_stable)
np.save(TABLES/'eligible_raw_oof.npy',raw_eligible)
pd.DataFrame(model_rows).to_csv(AUDIT/'model_inventory.csv',index=False)
pd.DataFrame(history_rows).to_csv(TABLES/'training_history.csv',index=False)

In [ ]:
# ============================================================
# 8. Fold-specific cross-fitted isotonic calibration
# ============================================================

p_stable = np.full(len(stable),np.nan,dtype=np.float64)
p_eligible = np.full(len(eligible),np.nan,dtype=np.float64)
calibrator_rows = []

for fold_id in sorted(stable.fold.unique()):
    train_idx = np.where(stable.fold.to_numpy() != fold_id)[0]
    test_idx = np.where(stable.fold.to_numpy() == fold_id)[0]
    eligible_idx = np.where(eligible.fold.to_numpy() == fold_id)[0]

    calibrator = IsotonicRegression(out_of_bounds='clip').fit(
        raw_stable[train_idx],
        stable_label[train_idx],
    )
    p_stable[test_idx] = calibrator.predict(raw_stable[test_idx])
    p_eligible[eligible_idx] = calibrator.predict(raw_eligible[eligible_idx])

    path = CALIBRATORS / f'fold{int(fold_id)}_seed{SEED}_annotation_isotonic.joblib'
    joblib.dump(calibrator,path)
    payload = {'x':calibrator.X_thresholds_.tolist(),'y':calibrator.y_thresholds_.tolist()}
    sidecar = {
        'fold': int(fold_id),
        'target': 'ANNOTATION_SAFE_TO_CLEAN',
        'method': 'IsotonicRegression',
        'training_rule': 'stable OOF rows outside held-out fold',
        'semantic_sha256': object_hash(payload),
        'joblib_sha256': sha256_file(path),
        **payload,
    }
    (CALIBRATORS/f'fold{int(fold_id)}_seed{SEED}_annotation_isotonic.json').write_text(
        json.dumps(sidecar,indent=2)
    )
    replay = joblib.load(path).predict(raw_eligible[eligible_idx])
    require(np.array_equal(replay,p_eligible[eligible_idx]),
            f'Fold {fold_id}: calibrator reload mismatch')
    calibrator_rows.append({
        'fold': int(fold_id),
        'joblib': str(path),
        'joblib_sha256': sha256_file(path),
        'semantic_sha256': object_hash(payload),
        'exact_reload_replay': True,
    })

require(np.isfinite(p_stable).all(), 'Incomplete stable calibrated probabilities.')
require(np.isfinite(p_eligible).all(), 'Incomplete eligible calibrated probabilities.')

np.save(TABLES/'stable_calibrated_oof.npy',p_stable)
np.save(TABLES/'eligible_calibrated_oof.npy',p_eligible)
pd.DataFrame(calibrator_rows).to_csv(AUDIT/'calibrator_inventory.csv',index=False)
print('Calibrators persisted:',len(calibrator_rows))

In [ ]:
# ============================================================
# 9. Conservative CLEAN/KEEP/UNCERTAIN policy and threshold sweep
# ============================================================

predictions = eligible.copy()
predictions['raw_p_annotation'] = raw_eligible
predictions['p_annotation_a5'] = p_eligible


def apply_minimum_run(frame, provisional, minimum_run=2):
    final = np.asarray(provisional,dtype=object).copy()
    demoted = 0
    for page, group in frame.groupby('page',sort=False):
        indices = group.index.to_numpy()
        shape = runtime[page]['observed'].shape
        mask = np.zeros(shape,np.uint8)
        selected_idx = indices[provisional[indices]=='CLEAN']
        if not len(selected_idx):
            continue
        mask[
            frame.loc[selected_idx,'y'].to_numpy(int),
            frame.loc[selected_idx,'x'].to_numpy(int),
        ] = 1
        n, labels, stats, _ = cv2.connectedComponentsWithStats(mask,8)
        for cid in range(1,n):
            if int(stats[cid,cv2.CC_STAT_AREA]) < minimum_run:
                ys,xs = np.where(labels==cid)
                coords = {(int(y),int(x)) for y,x in zip(ys,xs)}
                drop = [i for i in selected_idx if (int(frame.at[i,'y']),int(frame.at[i,'x'])) in coords]
                final[drop] = 'UNCERTAIN'
                demoted += len(drop)
    return final, demoted


def decisions_at_threshold(probability, threshold):
    # Only extreme annotation probability receives CLEAN.
    # Extreme print probability receives explicit KEEP.
    # Everything else remains UNCERTAIN and therefore A1.
    probability = np.asarray(probability,float)
    out = np.full(len(probability),'UNCERTAIN',object)
    out[probability <= 1-threshold] = 'KEEP'
    out[probability >= threshold] = 'CLEAN'
    return out


def policy_metrics(frame, decisions, threshold, demotions):
    work = frame.copy()
    work['decision'] = decisions
    clean = work.decision == 'CLEAN'
    tp_cleanup = int((clean & (work.label==LABEL_ANNOTATION)).sum())
    print_loss = int((clean & (work.label==LABEL_PRINT)).sum())
    uncertain_changed = int((clean & (work.label==LABEL_UNCERTAIN)).sum())

    component_rows=[]
    for page, g in work[clean].groupby('page'):
        auth = runtime[page]['reviewed_mask']
        _, labels, stats, _ = cv2.connectedComponentsWithStats(auth.astype(np.uint8),8)
        cids = labels[g.y.to_numpy(int),g.x.to_numpy(int)]
        g = g.copy(); g['component_id'] = cids
        for cid,q in g[g.component_id>0].groupby('component_id'):
            ann = int((q.label==LABEL_ANNOTATION).sum())
            prn = int((q.label==LABEL_PRINT).sum())
            unc = int((q.label==LABEL_UNCERTAIN).sum())
            catastrophic = bool(prn>=50 or (prn/max(ann,1)>1.0 and prn>=20))
            component_rows.append({
                'threshold':threshold,'page':page,'source_group':q.source_group.iloc[0],
                'component_id':int(cid),'selected':len(q),'annotation_cleaned':ann,
                'print_lost':prn,'truth_uncertain_changed':unc,'catastrophic':catastrophic,
            })
    components = pd.DataFrame(component_rows)
    catastrophes = int(components.catastrophic.sum()) if len(components) else 0

    page_rows=[]
    for page,g in work.groupby('page'):
        selected = g.decision=='CLEAN'
        ann = int((selected&(g.label==LABEL_ANNOTATION)).sum())
        prn = int((selected&(g.label==LABEL_PRINT)).sum())
        unc = int((selected&(g.label==LABEL_UNCERTAIN)).sum())
        cats = int(((components.page==page)&components.catastrophic).sum()) if len(components) else 0
        page_rows.append({
            'threshold':threshold,'page':page,'source_group':g.source_group.iloc[0],
            'eligible':len(g),'selected_clean':int(selected.sum()),
            'annotation_cleaned':ann,'print_lost':prn,'truth_uncertain_changed':unc,
            'utility':ann-5*prn-500*cats,'catastrophic_components':cats,
            'clean_coverage':float(selected.mean()),
            'uncertain_rate':float((g.decision=='UNCERTAIN').mean()),
        })
    pages = pd.DataFrame(page_rows)

    rng = np.random.default_rng(SEED)
    page_ids = pages.page.to_numpy()
    indexed = pages.set_index('page')
    boot=[]
    for _ in range(BOOTSTRAP_RESAMPLES):
        q=indexed.loc[rng.choice(page_ids,size=len(page_ids),replace=True)]
        ba=int(q.annotation_cleaned.sum()); bp=int(q.print_lost.sum()); bc=int(q.catastrophic_components.sum())
        boot.append((ba-5*bp-500*bc,bp/max(ba,1),bp/max(ba+bp,1)))
    boot=np.asarray(boot,float)

    summary={
        'threshold':float(threshold),
        'eligible_pixels':len(work),
        'selected_clean_pixels':int(clean.sum()),
        'annotation_cleaned':tp_cleanup,
        'print_lost':print_loss,
        'truth_uncertain_changed':uncertain_changed,
        'print_loss_per_annotation_cleaned':print_loss/max(tp_cleanup,1),
        'false_cleanup_rate':print_loss/max(tp_cleanup+print_loss,1),
        'utility':tp_cleanup-5*print_loss-500*catastrophes,
        'catastrophic_components':catastrophes,
        'benefiting_pages':int((pages.annotation_cleaned>=10).sum()),
        'benefiting_source_groups':int(pages.loc[pages.annotation_cleaned>=10,'source_group'].nunique()),
        'clean_coverage':float(clean.mean()),
        'uncertain_rate':float((work.decision=='UNCERTAIN').mean()),
        'minimum_run_demotions':int(demotions),
        'bootstrap_utility_95_low':float(np.quantile(boot[:,0],.025)),
        'bootstrap_utility_95_high':float(np.quantile(boot[:,0],.975)),
        'bootstrap_print_loss_per_annotation_95_high':float(np.quantile(boot[:,1],.975)),
        'bootstrap_false_cleanup_95_high':float(np.quantile(boot[:,2],.975)),
    }
    return summary,pages,components,work

sweep_summaries=[]; sweep_pages=[]; sweep_components=[]; policy_outputs={}
for threshold in CLEAN_THRESHOLDS:
    provisional = decisions_at_threshold(predictions.p_annotation_a5.to_numpy(),threshold)
    final,demotions = apply_minimum_run(predictions,provisional,MIN_CLEAN_RUN)
    summary,pages_df,components_df,work = policy_metrics(predictions,final,threshold,demotions)
    sweep_summaries.append(summary); sweep_pages.append(pages_df)
    if len(components_df): sweep_components.append(components_df)
    policy_outputs[threshold] = work

sweep = pd.DataFrame(sweep_summaries).sort_values('threshold',ascending=False)
sweep.to_csv(TABLES/'a5_threshold_sweep.csv',index=False)
pd.concat(sweep_pages,ignore_index=True).to_csv(TABLES/'a5_threshold_sweep_by_page.csv',index=False)
(pd.concat(sweep_components,ignore_index=True) if sweep_components else pd.DataFrame()).to_csv(
    TABLES/'a5_threshold_sweep_components.csv',index=False
)

primary = policy_outputs[PRIMARY_CLEAN_THRESHOLD].copy()
primary.to_parquet(TABLES/'a5_primary_oof_predictions.parquet',index=False)
primary.to_csv(TABLES/'a5_primary_oof_predictions.csv',index=False)
print(sweep.to_string(index=False))

In [ ]:
# ============================================================
# 10. Assemble complete A5 OOF pages from exact A1 fallback
# ============================================================

primary_page_metrics = []

for page, group in primary.groupby('page'):
    z = runtime[page]
    observed = z['observed']
    a1 = z['a1']
    paper = z['paper']
    authority = z['reviewed_mask']

    clean_rows = group[group.decision=='CLEAN']
    clean_mask = np.zeros(observed.shape,bool)
    if len(clean_rows):
        clean_mask[
            clean_rows.y.to_numpy(int),
            clean_rows.x.to_numpy(int),
        ] = True

    a5 = a1.copy()
    a5[clean_mask] = paper[clean_mask]

    outside = int(np.sum(a5[~authority] != observed[~authority]))
    nonclean = int(np.sum(a5[~clean_mask] != a1[~clean_mask]))
    require(outside==0, f'{page}: outside-mask drift={outside}')
    require(nonclean==0, f'{page}: non-CLEAN regression from A1={nonclean}')

    d=PAGES/page
    d.mkdir(parents=True,exist_ok=False)
    cv2.imwrite(str(d/'observed.png'),observed)
    cv2.imwrite(str(d/'A1.png'),a1)
    cv2.imwrite(str(d/'A5.png'),a5)
    cv2.imwrite(str(d/'clean_decision.png'),clean_mask.astype(np.uint8)*255)
    diff=np.zeros_like(observed); diff[a5!=a1]=255
    cv2.imwrite(str(d/'A5_minus_A1.png'),diff)

    metric = {
        'page':page,
        'source_group':group.source_group.iloc[0],
        'eligible':len(group),
        'selected_clean':int(clean_mask.sum()),
        'annotation_cleaned':int(((group.decision=='CLEAN')&(group.label==LABEL_ANNOTATION)).sum()),
        'print_lost':int(((group.decision=='CLEAN')&(group.label==LABEL_PRINT)).sum()),
        'truth_uncertain_changed':int(((group.decision=='CLEAN')&(group.label==LABEL_UNCERTAIN)).sum()),
        'outside_mask_drift':outside,
        'nonclean_a1_regression':nonclean,
        'observed_sha256':sha256_file(d/'observed.png'),
        'a1_sha256':sha256_file(d/'A1.png'),
        'a5_sha256':sha256_file(d/'A5.png'),
    }
    primary_page_metrics.append(metric)

primary_page_df=pd.DataFrame(primary_page_metrics)
primary_page_df.to_csv(TABLES/'a5_primary_results_by_page.csv',index=False)
primary_group_df=primary_page_df.groupby('source_group',as_index=False).agg(
    pages=('page','nunique'),eligible=('eligible','sum'),selected_clean=('selected_clean','sum'),
    annotation_cleaned=('annotation_cleaned','sum'),print_lost=('print_lost','sum'),
    truth_uncertain_changed=('truth_uncertain_changed','sum'),
)
primary_group_df['print_loss_per_annotation_cleaned'] = (
    primary_group_df.print_lost/primary_group_df.annotation_cleaned.clip(lower=1)
)
primary_group_df.to_csv(TABLES/'a5_primary_results_by_source_group.csv',index=False)
print('A5 whole pages:',len(primary_page_df))

In [ ]:
# ============================================================
# 11. Visual audit queues and contact sheets
# ============================================================

rank = primary_page_df.copy()
rank['risk_score'] = rank.print_lost*1000 + rank.truth_uncertain_changed*100 + rank.annotation_cleaned
rank.sort_values('risk_score',ascending=False).to_csv(TABLES/'a5_visual_review_queue.csv',index=False)


def rgb(gray):
    return cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)


def overlay(gray, mask, colour, alpha=.55):
    base=rgb(gray); out=base.copy(); m=np.asarray(mask,bool); c=np.asarray(colour,np.float32)
    out[m]=np.clip((1-alpha)*base[m]+alpha*c,0,255).astype(np.uint8)
    return out


def contact_sheet(rows,name,max_pages=16):
    rows=rows.head(max_pages)
    if not len(rows):
        return
    fig,axes=plt.subplots(len(rows),4,figsize=(20,max(5,5*len(rows))),squeeze=False)
    for i,(_,row) in enumerate(rows.iterrows()):
        page=str(row.page); d=PAGES/page
        obs=read_gray(d/'observed.png'); a1=read_gray(d/'A1.png'); a5=read_gray(d/'A5.png')
        clean_mask=read_gray(d/'clean_decision.png')>0
        panels=[
            (rgb(obs),'Observed'),
            (rgb(a1),'A1 fallback'),
            (rgb(a5),'A5 candidate'),
            (overlay(obs,clean_mask,(255,215,0)),f'CLEAN selected: {int(clean_mask.sum()):,}'),
        ]
        for ax,(image,title) in zip(axes[i],panels):
            ax.imshow(image); ax.set_title(f'{page} — {title}'); ax.axis('off')
    plt.tight_layout(); plt.savefig(VISUALS/f'{name}.png',dpi=130,bbox_inches='tight'); plt.close(fig)

contact_sheet(rank.sort_values(['print_lost','selected_clean'],ascending=False),'most_print_loss')
contact_sheet(rank.sort_values(['annotation_cleaned','print_lost'],ascending=[False,True]),'most_annotation_cleanup')
contact_sheet(rank.sort_values('selected_clean',ascending=False),'most_changed')

plt.figure(figsize=(9,7))
plt.plot(sweep.print_loss_per_annotation_cleaned,sweep.clean_coverage,marker='o')
for row in sweep.itertuples():
    plt.text(row.print_loss_per_annotation_cleaned,row.clean_coverage,' '+str(row.threshold))
plt.xlabel('Print pixels lost per annotation pixel cleaned')
plt.ylabel('Eligible CLEAN coverage')
plt.title('A5 development safety/coverage frontier')
plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(VISUALS/'a5_safety_coverage_frontier.png',dpi=180); plt.close()

print('Visuals:',VISUALS)

In [ ]:
# ============================================================
# 12. Development verdict, identity and durable training bundle
# ============================================================

primary_summary = sweep.loc[sweep.threshold==PRIMARY_CLEAN_THRESHOLD].iloc[0].to_dict()

# Development evidence flags. These do not authorise promotion.
gates = {
    'integrity_finite_predictions': bool(np.isfinite(raw_eligible).all() and np.isfinite(p_eligible).all()),
    'source_group_oof_no_overlap': True,
    'outside_mask_immutability': bool((primary_page_df.outside_mask_drift==0).all()),
    'nonclean_a1_equivalence': bool((primary_page_df.nonclean_a1_regression==0).all()),
    'zero_catastrophic_components': bool(primary_summary['catastrophic_components']==0),
    'positive_utility': bool(primary_summary['utility']>0),
    'positive_clustered_utility': bool(primary_summary['bootstrap_utility_95_low']>0),
    'nontrivial_cleanup': bool(primary_summary['annotation_cleaned']>=500 and primary_summary['benefiting_pages']>=10),
    'visual_audit_complete': False,
}

first_failed = next((name for name,value in gates.items() if value is False),None)
verdict = {
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'notebook':'88_A5_Contextual_Safe_Cleanup_Model_Training',
    'development_only':True,
    'independent_validation':False,
    'production_authorisation':False,
    'source_notebook87_run':str(SOURCE_N87),
    'primary_threshold':PRIMARY_CLEAN_THRESHOLD,
    'primary_summary':primary_summary,
    'gates':gates,
    'first_failed_or_incomplete_gate':first_failed,
    'decision':(
        'DEVELOPMENT_EVIDENCE_READY_FOR_VISUAL_REVIEW'
        if first_failed=='visual_audit_complete'
        else 'RETAIN_AS_DEVELOPMENT_RESEARCH'
    ),
}
(RUN/'notebook88_verdict.json').write_text(json.dumps(verdict,indent=2,default=str))

software={
    'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,
    'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,
    'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform(),
}
(RUN/'software_environment.json').write_text(json.dumps(software,indent=2))

identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'notebook88-a5-contextual-safe-cleanup-development',
    'source_notebook87_run':str(SOURCE_N87),
    'source_prediction_lock_sha256':sha256_file(SOURCE_N87/'prediction_lock.json'),
    'source_truth_table_sha256':sha256_file(SOURCE_N87/'tables'/'paired_hitl_ambiguous_truth.parquet'),
    'review_manifest_sha256':sha256_file(review_manifest_path),
    'channel_schema_sha256':sha256_file(INPUTS/'channel_schema.json'),
    'fold_map_sha256':sha256_file(INPUTS/'fold_map.csv'),
    'seed':SEED,'patch':PATCH,'channels':CHANNELS,'batch':BATCH,'epochs':EPOCHS,
    'learning_rate':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,
    'clean_thresholds':CLEAN_THRESHOLDS,'primary_clean_threshold':PRIMARY_CLEAN_THRESHOLD,
    'minimum_clean_run':MIN_CLEAN_RUN,'models':model_rows,'calibrators':calibrator_rows,
    'truth_in_inference_channels':False,'independent_data_used':False,
    'software':software,
}
(RUN/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

lines=[
    'NOTEBOOK 88 — A5 CONTEXTUAL SAFE-CLEANUP MODEL TRAINING','',
    f'Source Notebook 87 run: {SOURCE_N87}',
    f'Pages: {primary_page_df.page.nunique()}',
    f'Source groups: {primary_page_df.source_group.nunique()}',
    f'Eligible pixels: {len(primary)}',
    f'Primary CLEAN threshold: {PRIMARY_CLEAN_THRESHOLD}',
    f'Pixels selected to clean: {int(primary_summary["selected_clean_pixels"])}',
    f'Annotation pixels cleaned: {int(primary_summary["annotation_cleaned"])}',
    f'Genuine-print pixels lost: {int(primary_summary["print_lost"])}',
    f'Truth-uncertain pixels changed: {int(primary_summary["truth_uncertain_changed"])}',
    f'Utility: {primary_summary["utility"]}',
    f'Catastrophic components: {int(primary_summary["catastrophic_components"])}',
    f'CLEAN coverage: {primary_summary["clean_coverage"]:.6f}',
    f'UNCERTAIN rate: {primary_summary["uncertain_rate"]:.6f}','',
    'DEVELOPMENT FLAGS',
] + [f'- {k}: {v}' for k,v in gates.items()] + [
    '',
    'Interpretation: development OOF evidence only. No automatic promotion is authorised.',
    'A5 defaults to exact A1 wherever CLEAN is not validly emitted.',
]
(RUN/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
# ============================================================
# 13. Final upload package
# ============================================================

UPLOAD_ARTEFACTS = [
    RUN/'executive_summary.txt',
    RUN/'notebook88_verdict.json',
    RUN/'experiment_identity.json',
    RUN/'software_environment.json',
    INPUTS/'channel_schema.json',
    INPUTS/'fold_map.csv',
    INPUTS/'page_role_manifest.csv',
    TABLES/'a5_threshold_sweep.csv',
    TABLES/'a5_primary_results_by_page.csv',
    TABLES/'a5_primary_results_by_source_group.csv',
    TABLES/'a5_visual_review_queue.csv',
    AUDIT/'source_group_split_audit.csv',
    AUDIT/'input_and_label_audit.csv',
    AUDIT/'model_inventory.csv',
    AUDIT/'calibrator_inventory.csv',
    VISUALS/'a5_safety_coverage_frontier.png',
    VISUALS/'most_print_loss.png',
    VISUALS/'most_annotation_cleanup.png',
    VISUALS/'most_changed.png',
]

for source in UPLOAD_ARTEFACTS:
    require(source.exists(), f'Missing upload artefact: {source}')
    shutil.copy2(source,UPLOAD/source.name)

manifest_rows=[]
for path in sorted(UPLOAD.iterdir()):
    if path.is_file():
        manifest_rows.append({
            'file':path.name,'bytes':path.stat().st_size,'sha256':sha256_file(path)
        })
pd.DataFrame(manifest_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)

readme = [
    'NOTEBOOK 88 A5 CONTEXTUAL SAFE-CLEANUP REVIEW PACKAGE','',
    'Start with executive_summary.txt and notebook88_verdict.json.',
    'Inspect a5_threshold_sweep.csv, page/source-group tables and the three contact sheets.',
    'Models and calibrators remain in the full run directory and are identified by experiment_identity.json.',
    'This package is development evidence only.',
]
(UPLOAD/'README.txt').write_text('\n'.join(readme))

zip_base=RUN/'notebook88_a5_contextual_safe_cleanup_bundle'
shutil.make_archive(str(zip_base),'zip',UPLOAD)
print('Bundle:',str(zip_base)+'.zip')
print(json.dumps(verdict,indent=2,default=str))